# Quantitative MPC speech report

Set the explicit input dataset and output directory below, then run all cells. No LLM calls or changes to the classification outputs are made. Each CSV row is counted once; duplicate IDs and missing metadata are flagged rather than silently repaired.

Primary counts include an Unclassified bucket. Secondary counts exclude the primary theme, count a theme at most once per speech and exclude rows without a primary classification. Secondary totals can exceed the number of speeches.

Internal/external status requires an ISO speech date and a match in the dated membership reference. Unknown speakers, missing dates and conflicting date/year metadata stay unresolved. Outside recorded tenure does not prove non-membership: check the reference for missing terms or reappointments.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'quantify_mpc_speeches.py').exists():
    raise FileNotFoundError('Run this notebook with the speech-analysis-project folder as the working directory.')

from quantify_mpc_speeches import run_analysis

# Change these paths when your new merged dataset is ready.
DATABASE_CSV = PROJECT_ROOT / 'outputs' / 'full_run' / 'mpc_speech_database_with_llm.csv'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'full_run' / 'quantitative'
MEMBERSHIP_CSV = PROJECT_ROOT / 'mpc_membership.csv'

summary = run_analysis(DATABASE_CSV, OUTPUT_ROOT, MEMBERSHIP_CSV)
print(json.dumps(summary, indent=2))

def report(name):
    return pd.read_csv(OUTPUT_ROOT / f'{name}.csv', keep_default_na=False)

def save_chart(fig, name):
    fig.tight_layout()
    fig.savefig(OUTPUT_ROOT / f'{name}.png', dpi=150, bbox_inches='tight')
    plt.show()


## 1. Speeches per year
Counts reflect the supplied corpus, not necessarily all speeches delivered in that year. Unknown years remain visible.


In [ ]:
by_year = report('speeches_by_year')
display(by_year)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(by_year['year'].astype(str), by_year['speech_count'])
ax.set(title='Speeches per year', xlabel='Year', ylabel='Speech count')
save_chart(fig, 'speeches_by_year')


## 2. Themes: LLM versus keyword method
The keyword method is the existing keyword-frequency classification, not total speech length. All ten taxonomy themes are shown, including zero counts. Percentages in the CSV use all input speech rows as the denominator; they are not LLM confidence scores.


In [ ]:
themes = report('theme_counts')
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for role, ax in zip(['primary', 'secondary'], axes):
    table = themes[themes['theme_role'] == role].pivot(index='theme_label', columns='method', values='speech_count').fillna(0).astype(int)
    display(table.rename_axis(f'{role.title()} theme'))
    table.plot.barh(ax=ax)
    ax.set(title=f'{role.title()} themes', xlabel='Speech count', ylabel='')
save_chart(fig, 'theme_counts')


## 3. Speeches by speaker
Known name aliases are consolidated. Unrecognised labels are retained for review, without guessing a name from event text.


In [ ]:
speakers = report('speeches_by_speaker').sort_values('speech_count')
display(speakers.sort_values('speech_count', ascending=False))
fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(speakers))))
ax.barh(speakers['speaker'], speakers['speech_count'])
ax.set(title='Speeches by speaker', xlabel='Speech count')
save_chart(fig, 'speeches_by_speaker')


## 4. Internal versus external MPC members at the speech date
Unresolved and outside-reference cases are shown separately, not allocated to internal or external members.


In [ ]:
membership = report('speeches_by_mpc_member_type')
display(membership)
display(report('speeches_by_mpc_member_type_year'))
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(membership['mpc_member_type'].str.replace('_', ' '), membership['speech_count'])
ax.set(title='MPC membership status at speech date', xlabel='Speech count')
save_chart(fig, 'speeches_by_mpc_member_type')


## Speaker/topic cross-tab and quality review
Use METHOD and THEME_ROLE to select LLM/keyword and primary/secondary themes. The CSV reports also provide theme counts by year and MPC member type.


In [ ]:
METHOD = 'llm'
THEME_ROLE = 'primary'
by_speaker_theme = report('theme_counts_by_speaker')
selected = by_speaker_theme[(by_speaker_theme['method'] == METHOD) & (by_speaker_theme['theme_role'] == THEME_ROLE)]
cross_tab = selected.pivot(index='speaker', columns='theme_label', values='speech_count').fillna(0).astype(int)
display(cross_tab)
quality = report('data_quality_issues')
display(quality)
print(f'All tables and charts saved to: {OUTPUT_ROOT}')
